# Content Based Filtering



Recommendation systems are a collection of algorithms used to recommend items to users based on information taken from the user. These systems have become ubiquitous, and can be commonly seen in online stores, movies databases and job finders. In this notebook, we will explore Content-based recommendation systems and implement a simple version of one using Python and the Pandas library.


### Table of contents

<div class="alert alert-block alert-info" style="margin-top: 20px">
    <ol>
        <li><a href="https://#ref1">Acquiring the Data</a></li>
        <li><a href="https://#ref2">Preprocessing</a></li>
        <li><a href="https://#ref3">Content-Based Filtering</a></li>
    </ol>
</div>
<br>


<a id="ref1"></a>

# Acquiring the Data


In [2]:
import pandas as pd
import os

In [15]:
#Storing the movie information into a pandas dataframe
movies_df = pd.read_csv('movies.csv')
#Storing the user information into a pandas dataframe
# ratings_df = pd.read_csv('ratings.csv')

movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [16]:
dataset_path = os.path.join(os.path.expanduser('~'), 'datasets', 'movies', '7')

credits_df = pd.read_csv(dataset_path + '/credits.csv')
# movies_meta_df = pd.read_csv(dataset_path + '/movies_metadata.csv')
ratings_df = pd.read_csv(dataset_path + '/ratings_small.csv')

/var/folders/lr/44zcpqfx2196g_qy5t3kc7m40000gp/T/ipykernel_2142/4238524136.py:4: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_meta_df = pd.read_csv(dataset_path + '/movies_metadata.csv')


In [83]:
# Dropping useless bits from movies_meta_df
movies_df = pd.read_csv(f"{dataset_path}/movies_metadata.csv", usecols=['id', 'title', 'release_date', 'genres', 'popularity', 'vote_average', 'vote_count'])

/var/folders/lr/44zcpqfx2196g_qy5t3kc7m40000gp/T/ipykernel_2142/1038718327.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_df = pd.read_csv(f"{dataset_path}/movies_metadata.csv", usecols=['id', 'title', 'release_date', 'genres', 'popularity', 'vote_average', 'vote_count'])


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ratings_small_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/7/ratings_small.csv')

In [ ]:
# movies_meta_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/7/movies_metadata.csv')
# movies_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/movies/movies.csv')

In [ ]:
# ratings_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/movies/ratings.csv')

In [ ]:
# credits_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Datasets/movie_datasets/7/credits.csv')
# credits_df.head()

In [84]:
movies_df = movies_df.sort_values(by='release_date', ascending=False)
# movies_meta_df.head(3)
movies_df = movies_df.drop(35587) # A weird film entry is now gone!
movies_df.head(3)

,genres,id,popularity,release_date,title,vote_average,vote_count
26559,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",76600,6.020055,2020-12-16,Avatar 2,0.0,58.0
38885,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",299782,0.238154,2018-12-31,The Other Side of the Wind,0.0,1.0
30402,"[{'id': 53, 'name': 'Thriller'}, {'id': 28, 'n...",38700,2.178546,2018-11-07,Bad Boys for Life,0.0,12.0


In [85]:
for index, value in movies_df["genres"].head(10).items():
    print(f"Index {index}: {value}")

Index 26559: [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 878, 'name': 'Science Fiction'}]
Index 38885: [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'name': 'Drama'}]
Index 30402: [{'id': 53, 'name': 'Thriller'}, {'id': 28, 'name': 'Action'}, {'id': 80, 'name': 'Crime'}]
Index 38130: [{'id': 18, 'name': 'Drama'}, {'id': 10749, 'name': 'Romance'}]
Index 44535: [{'id': 18, 'name': 'Drama'}]
Index 33359: [{'id': 28, 'name': 'Action'}, {'id': 35, 'name': 'Comedy'}, {'id': 14, 'name': 'Fantasy'}, {'id': 878, 'name': 'Science Fiction'}]
Index 33358: [{'id': 16, 'name': 'Animation'}, {'id': 12, 'name': 'Adventure'}, {'id': 35, 'name': 'Comedy'}, {'id': 28, 'name': 'Action'}, {'id': 10751, 'name': 'Family'}]
Index 30536: [{'id': 14, 'name': 'Fantasy'}, {'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]
Index 44301: [{'id': 35, 'name': 'Comedy'}, {'id': 10402, 'name': 'Music'}, {'id': 10749, 'name': 'Romance'}, {'id': 878, 'name'

<a id="ref2"></a>

# Preprocessing


In [55]:

from math import sqrt
import numpy as np
import matplotlib.pyplot as plt

import ast
%matplotlib inline

First, let's get all of the imports out of the way:


Now let's read each file into their Dataframes:


In [41]:
# Make a genres table / format it to be able to use it as a feature


In [27]:
# prompt: print the dimensions of ratings, credits, movies, movies_metadata

print("ratings_df dimensions:", ratings_df.shape)
print("credits_df dimensions:", credits_df.shape)
print("movies_df dimensions:", movies_df.shape)
# print("movies_meta_df dimensions:", movies_meta_df.shape)


ratings_df dimensions: (100004, 4)
credits_df dimensions: (45476, 3)
movies_df dimensions: (34208, 3)
movies_meta_df dimensions: (45465, 24)


In [ ]:


def get_first_3_cast(cast_series):
    """
    Extracts names and IDs of the first 3 cast members from a Pandas Series.

    Args:
        cast_series (pandas.Series): A Pandas Series containing cast information
                                      (string representation of list of dictionaries).

    Returns:
        pandas.DataFrame: DataFrame with a column 'cast_info' containing tuples of (name, ID)
                         for the first 3 cast members.
    """

    all_cast_info = []

    for cast_list_str in cast_series:
        cast_list = ast.literal_eval(cast_list_str)  # Convert string to list
        cast_info = []
        for i in range(min(3, len(cast_list))):
            cast_info.append((cast_list[i]['name'], cast_list[i]['id']))  # Create (name, ID) tuple
        all_cast_info.append(cast_info)

    return pd.DataFrame({'cast_info': all_cast_info})


# Usage:
cast_info_df = get_first_3_cast(credits_df['cast'])
cast_info_df.head()

In [ ]:
def get_directors_from_crew(credits):
    """
    Extracts the names of the director(s) from the 'crew' column of a Pandas DataFrame.

    Args:
        credits (pandas.DataFrame): A Pandas DataFrame containing 'crew' column with crew information.

    Returns:
        pandas.DataFrame: DataFrame with a column 'director_name' containing the names of the director(s), and the corresponding director 'id', and 'credits id' for the film.

    """
    # Set 'id' column as index
    global director_credit_id
    credits = credits.set_index('id')

    director_info = []

    for index, row in credits.iterrows():  # Iterate using index and row
        crew_list_str = row['crew']
        crew_list = ast.literal_eval(crew_list_str)
        director_name = None
        director_id = None
        film_id = index  # Get the film 'id' (now index)

        for crew_member_l in crew_list:
            if crew_member_l['job'] == 'Director':
                director_name = crew_member_l['name']
                director_credit_id = crew_member_l['credit_id']
                break

        director_info.append((director_name, director_credit_id, film_id))

    return pd.DataFrame({'director_info': director_info})


director_info_df = get_directors_from_crew(credits_df)
director_info_df.head()


In [ ]:
def get_first_3_crew(crew_series):
    """
    Extracts names and IDs of the first 3 crew members from a Pandas Series.

    Args:
        crew_series (pandas.Series): A Pandas Series containing crew information
                                      (string representation of list of dictionaries).

    Returns:
        pandas.DataFrame: DataFrame with columns 'crew_name' and 'crew_id' for the first 3 crew members.
    """

    all_crew_info = []

    for crew_list_str in crew_series:
        crew_list_s = ast.literal_eval(crew_list_str)  # Convert string to list
        crew_info = []
        for i in range(min(3, len(crew_list_s))):
            crew_info.append((crew_list_s[i]['name'], crew_list_s[i]['job'], crew_list_s[i]['id']))  # Create (name, job, ID) tuple
        all_crew_info.append(crew_info)

    return pd.DataFrame({'crew_info': all_crew_info})


crew_info_df = get_first_3_crew(credits_df['crew'])
# crew_info_df.head()

In [32]:
# find all the unique jobs of from crew_info_df
# dataframe example: [(John Lasseter, Director, 7879), (Joss Whedon...
unique_jobs = set()
for crew_list in crew_info_df['crew_info']:
    for crew_member in crew_list:
        unique_jobs.add(crew_member[1])

In [54]:
top_3_credits_df = pd.concat([cast_info_df, director_info_df, credits_df['id']], axis=1)
# top_3_credits_df = pd.concat([cast_info_df, director_info_df, credits_df['id']], axis=1)
# top_3_credits_df.head()
top_3_credits_df.head()

,cast_info,director_info,id
0,"[(Tom Hanks, 31), (Tim Allen, 12898), (Don Ric...","(John Lasseter, 52fe4284c3a36847f8024f49, 862)",862
1,"[(Robin Williams, 2157), (Jonathan Hyde, 8537)...","(Joe Johnston, 52fe44bfc3a36847f80a7c7d, 8844)",8844
2,"[(Walter Matthau, 6837), (Jack Lemmon, 3151), ...","(Howard Deutch, 52fe466a9251416c75077a89, 15602)",15602
3,"[(Whitney Houston, 8851), (Angela Bassett, 978...","(Forest Whitaker, 52fe44779251416c91011acb, 31...",31357
4,"[(Steve Martin, 67773), (Diane Keaton, 3092), ...","(Charles Shyer, 52fe44959251416c75039eef, 11862)",11862


In [67]:
# movies_df.head()
# Make a separate table for genres with the genre name and the corresponding id. Take it from the genres column on movies_df
# Create a separate table for genres with the genre name and the corresponding id
import json

def extract_genres():
    genres_set = set()
    for genres_str in movies_df['genres']:
        genres_list = json.loads(genres_str.replace("'", "\""))
        for genre in genres_list:
            genres_set.add((genre['id'], genre['name']))
    return pd.DataFrame(genres_set, columns=['genre_id', 'genre_name'])

genres_df = extract_genres()
drop_entries = [7759, 7760, 7761, 11602, 11176, 33751, 29812, 2883]
genres_df = genres_df[~genres_df['genre_id'].isin(drop_entries)]
genres_df

,genre_id,genre_name
0,10751,Family
1,10402,Music
2,18,Drama
3,7759,GoHands
4,7760,BROSTA TV
5,80,Crime
6,37,Western
7,36,History
8,878,Science Fiction
9,10769,Foreign


In [113]:
movies_df 

,genres,id,popularity,release_date,title,vote_average,vote_count,genre_list
26559,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",76600,6.020055,2020-12-16,Avatar 2,0.0,58.0,"[Action, Adventure, Fantasy, Science Fiction]"
38885,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",299782,0.238154,2018-12-31,The Other Side of the Wind,0.0,1.0,"[Comedy, Drama]"
30402,"[{'id': 53, 'name': 'Thriller'}, {'id': 28, 'n...",38700,2.178546,2018-11-07,Bad Boys for Life,0.0,12.0,"[Thriller, Action, Crime]"
38130,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",332283,3.328261,2018-04-25,Mary Shelley,0.0,1.0,"[Drama, Romance]"
44535,"[{'id': 18, 'name': 'Drama'}]",412059,0.155147,2018-04-04,Mobile Homes,0.0,1.0,[Drama]
...,...,...,...,...,...,...,...,...
45148,[],438910,0.001586,NaN,Engineering Red,6.0,2.0,[]
45203,"[{'id': 9648, 'name': 'Mystery'}, {'id': 878, ...",433711,0.00022,NaN,All Superheroes Must Die 2: The Last Superhero,4.0,1.0,"[Mystery, Science Fiction]"
45338,[],335251,0.0,NaN,The Land Where the Blues Began,0.0,0.0,[]
45410,"[{'id': 18, 'name': 'Drama'}, {'id': 80, 'name...",449131,0.008903,NaN,Aprel,6.0,1.0,"[Drama, Crime]"


In [114]:
ohe_movies_df = movies_df

# Bad genres: (7759,GoHands), (7760,BROSTA TV), (7761,Mardock Scramble Production Committee), (11602,Vision View Entertainment), (11176,Carousel Productions), (33751,Sentai Filmworks), (29812,Telescene Film Group Productions)
drop_entries = [7759, 7760, 7761, 11602, 11176, 33751, 29812, 2883]
# Drop the genre entries with id's listed above
ohe_movies_df['genres'] = ohe_movies_df['genres'].apply(
    lambda x: [genre for genre in x if genre['id'] not in drop_entries] if isinstance(x, list) else []
)

# List all the genres (validation check - there's no dodgy genres left in)
all_genres = set()
for genres in ohe_movies_df['genres']:
    for genre in genres:
        all_genres.add(genre['name'])
print(all_genres)

{'Crime', 'Romance', 'Western', 'Comedy', 'Documentary', 'Family', 'Foreign', 'Animation', 'History', 'Horror', 'Science Fiction', 'TV Movie', 'Adventure', 'War', 'Fantasy', 'Thriller', 'Action', 'Music', 'Mystery', 'Drama'}


In [115]:
# Extract genre names
ohe_movies_df['genre_list'] = ohe_movies_df['genres'].apply(lambda x: [genre['name'] for genre in x] if isinstance(x, list) else [])
ohe_movies_df.head()

,genres,id,popularity,release_date,title,vote_average,vote_count,genre_list
26559,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",76600,6.020055,2020-12-16,Avatar 2,0.0,58.0,"[Action, Adventure, Fantasy, Science Fiction]"
38885,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",299782,0.238154,2018-12-31,The Other Side of the Wind,0.0,1.0,"[Comedy, Drama]"
30402,"[{'id': 53, 'name': 'Thriller'}, {'id': 28, 'n...",38700,2.178546,2018-11-07,Bad Boys for Life,0.0,12.0,"[Thriller, Action, Crime]"
38130,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",332283,3.328261,2018-04-25,Mary Shelley,0.0,1.0,"[Drama, Romance]"
44535,"[{'id': 18, 'name': 'Drama'}]",412059,0.155147,2018-04-04,Mobile Homes,0.0,1.0,[Drama]


In [116]:
from sklearn.preprocessing import MultiLabelBinarizer

# Initialize MultiLabelBinarizer
mlb = MultiLabelBinarizer()

genre_ohe = pd.DataFrame(mlb.fit_transform(ohe_movies_df['genre_list']),
                         columns=mlb.classes_,
                         index=ohe_movies_df.index)

# Merge back into sample_genres_movies_df
ohe_movies_df = pd.concat([ohe_movies_df, genre_ohe], axis=1)

# Drop unnecessary columns
ohe_movies_df = ohe_movies_df.drop(columns=['genres', 'genre_list'])

# print(sample_genres_movies_df.head())
ohe_movies_df

,id,popularity,release_date,title,vote_average,vote_count,Action,Adventure,Animation,Comedy,...,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
26559,76600,6.020055,2020-12-16,Avatar 2,0.0,58.0,1,1,0,0,...,0,0,0,0,0,1,0,0,0,0
38885,299782,0.238154,2018-12-31,The Other Side of the Wind,0.0,1.0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
30402,38700,2.178546,2018-11-07,Bad Boys for Life,0.0,12.0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
38130,332283,3.328261,2018-04-25,Mary Shelley,0.0,1.0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
44535,412059,0.155147,2018-04-04,Mobile Homes,0.0,1.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45148,438910,0.001586,NaN,Engineering Red,6.0,2.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
45203,433711,0.00022,NaN,All Superheroes Must Die 2: The Last Superhero,4.0,1.0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
45338,335251,0.0,NaN,The Land Where the Blues Began,0.0,0.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
45410,449131,0.008903,NaN,Aprel,6.0,1.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# ratings_df.shape

(22884377, 4)

In [ ]:
# userId_ratings = ratings_df.groupby('userId').size().reset_index(name='count')
# userId_ratings = userId_ratings.sort_values(by='userId', ascending=False)
# userId_ratings.head()


# Creating a user profile

In [56]:
user_ratings = """
710,golden eye, 4
1770,Michael Collins, 5
76600,Avatar 2, 3
141052,Justice League, 2.5
284053,Thor: Ragnarok, 4
341013,Atomic Blonde, 3.8
374720,Dunkirk, 4.7
339403,Baby Driver, 4.7
324852,Despicable Me 3, 4
419192,McLaren, 3.5
283995,Guardians of the Galaxy Vol. 2, 3.8
324849,The Lego Batman Movie, 4.5
324552,John Wick: Chapter 2, 3.5
180863,T2 Trainspotting, 3.6
318846,The Big Short, 4.4
406,La Haine, 4.65
424,Schindler's List, 4.8
627,Trainspotting, 4.9
100,"Lock, Stock and Two Smoking Barrels", 4.3
107,Snatch, 4.9
1429,25th Hour, 3.3
49517,Tinker Tailor Soldier Spy, 4.1
6538,Charlie Wilson's War, 3.9
10315,Fantastic Mr. Fox, 4.7
27205,Inception, 3.6
106646,The Wolf of Wall Street, 4.5
120467,The Grand Budapest Hotel, 4.6
157336,Interstellar, 3.9
261023,Black Mass, 3.6
278,The Shawshank Redemption, 3
4232,Scream, 2.4
1893,Star Wars: Episode I - The Phantom Menace, 3.8
550,Fight Club, 4.2
98,Gladiator, 4
508,Love Actually, 2.1
228967,The Interview, 2.7
1895,Star Wars: Episode III - Revenge of the Sith, 3.5
18785,The Hangover, 1.9
67913,The Guard, 3.7
40807,50/50, 2.5
70160,The Hunger Games, 2.4
77930,Magic Mike, 1
72190,World War Z, 2.5
187017,22 Jump Street, 1.9
198663,The Maze Runner, 2
207703,Kingsman: The Secret Service, 2.1
99861,Avengers: Age of Ultron, 2
167073,Brooklyn, 2.8
254470,Pitch Perfect 2, 1.9
271718,Trainwreck, 1
314365,Spotlight, 4.9
259693,The Conjuring 2, 2.3
308266,War Dogs, 2.1
324786,Hacksaw Ridge, 3.1
330459,Rogue One: A Star Wars Story, 2.9
339846,Baywatch, 2
"""

In [57]:
from io import StringIO


# Convert into a DataFrame
user_ratings_df = pd.read_csv(StringIO(user_ratings), header=None, names=["movieId", "movie_name", "rating"])

# Drop the movie_name column as it's not needed for appending to the ratings DataFrame
user_ratings_df = user_ratings_df.drop(columns=["movie_name"])

user_ratings_df['movieId'] = pd.to_numeric(user_ratings_df['movieId'], errors='coerce')
user_ratings_df = user_ratings_df.dropna(subset=['movieId'])
user_ratings_df['movieId'] = user_ratings_df['movieId'].astype(int)

# Testing user id = 999999
user_ratings_df["userId"] = 999999

# Reorder columns to match the existing ratings DataFrame
user_ratings_df = user_ratings_df[["userId", "movieId", "rating"]]


# Print the DataFrame
print(user_ratings_df.head(3))


   userId  movieId  rating
0  999999      710     4.0
1  999999     1770     5.0
2  999999    76600     3.0


In [59]:
ratings_df = pd.concat([ratings_df, user_ratings_df], ignore_index=True)
# ratings_df = ratings_df.drop('timestamp', axis=1) 
ratings_df.head()
# ratings_df.tail(10)


,userId,movieId,rating
0,1,31,2.5
1,1,1029,3.0
2,1,1061,3.0
3,1,1129,2.0
4,1,1172,4.0


In [ ]:
# user_id_to_find = 999999
# entries_with_user_id = ratings_df[ratings_df['userId'] == user_id_to_find]
# entries_with_user_id


In [ ]:

def create_user_profile(user_id, ratings_df, top_3_credits_df):
  """Creates a user profile based on ratings for movies with shared cast/directors."""

  user_ratings = ratings_df[ratings_df['userId'] == user_id]
  # print(user_ratings)
  profile = {}

  for _, rating_row in user_ratings.iterrows():
    movie_id = rating_row['movieId']
    rating = rating_row['rating']

    #find the index of the movie_id in top_3_credits_df
    try:
        movie_index = top_3_credits_df[top_3_credits_df['id'] == movie_id].index[0]
    except IndexError:
        continue # if the movie_id isn't in top_3_credits_df, skip this movie


    # Add cast and director IDs to user profile with weighted ratings
    cast_ids = top_3_credits_df['cast_info'].iloc[movie_index]
    for cast_member in cast_ids:
        profile[cast_member[1]] = profile.get(cast_member[1], 0) + rating

    director_info = top_3_credits_df['director_info'].iloc[movie_index]
    if director_info is not None:
      profile[director_info[1]] = profile.get(director_info[1], 0) + rating
          
    

  return profile


# Example usage:
user_id = 999999  # Replace with your desired user ID
user_profile = create_user_profile(user_id, ratings_df, top_3_credits_df)
# user_profile

# Generate recommendations


In [ ]:
def recommend_movies(user_id, user_profile, movies_df, top_3_credits_df, ratings_df, top_n=20):
  """Recommends movies based on user profile and movie cast/director."""

  user_rated_movies = set(ratings_df[ratings_df['userId'] == user_id]['movieId'])
  unrated_movies = movies_df[~movies_df['movieId'].isin(user_rated_movies)]
  recommendations = []

  for _, movie_row in unrated_movies.iterrows():
    movie_id = movie_row['movieId']
    score = 0

    try:
         movie_index = top_3_credits_df[top_3_credits_df['id'] == movie_id].index[0]
    except IndexError:
        continue # if the movie_id isn't in top_3_credits_df, skip this movie
        
    # Calculate weighted score based on user profile and movie cast/director, genres, 
    cast_ids = top_3_credits_df['cast_info'].iloc[movie_index]
    for cast_member in cast_ids:
        score += user_profile.get(cast_member[1], 0)

    director_info = top_3_credits_df['director_info'].iloc[movie_index]
    if director_info is not None:
      score += user_profile.get(director_info[1], 0)
    
    # Calculate weighted score based on genres
    genre_ids = movie_row['genres']
    for genre_id in genre_ids:
        score += user_profile.get(genre_id, 0)
    
    recommendations.append((movie_id, score))

  recommendations.sort(key=lambda x: x[1], reverse=True)  # Sort by score
  top_recommendations = recommendations[:top_n]


  # recommendations.sort(key=lambda x: x[1])  # Sort in ascending order
  lowest_recommendations = recommendations[-top_n:]
  least_rated_movie_ids = [movie_id for movie_id, score in lowest_recommendations]

  recommended_movie_ids = [movie_id for movie_id, score in top_recommendations]
  # least_rated_movie_ids = [movie_id for movie_id, score in recommendations[:top_n]]

  return recommended_movie_ids, least_rated_movie_ids

In [ ]:
recommended_movie_ids, least_rated_movie_ids = recommend_movies(user_id, user_profile, movies_df, top_3_credits_df, ratings_df)
print(movies_df[movies_df['movieId'].isin(recommended_movie_ids)][['title']])


                                                   title
234                                         Forget Paris
269                          Madness of King George, The
581                                                Ghost
809                                        Kaspar Hauser
840           Every Other Weekend (Un week-end sur deux)
1386         Message to Love: The Isle of Wight Festival
1580                                      Ice Storm, The
1811                               Six Days Seven Nights
1862                                   On the Waterfront
2129                          Man Who Knew Too Much, The
4321                        Cheech & Chong's Nice Dreams
4692                                     Little Man Tate
7889                    Plain Dirty (a.k.a. Briar Patch)
7983                               Bourne Supremacy, The
7999              Lancelot of the Lake (Lancelot du Lac)
8270                                      Falling Angels
9092   71 Fragments of a Chrono

In [ ]:
print(movies_df[movies_df['movieId'].isin(least_rated_movie_ids)][['title']])

                                                  title
33873                The Man Who Had His Hair Cut Short
33874                                  Frits and Franky
33888                                     De Vlaschaard
33890                                              Noah
33897                                 Morning Departure
33899                           Cool Cat Saves the Kids
33930                                       Aşk Kırmızı
33965                                         Unrelated
33969                                              Felt
34014                                       Trophy Kids
34032                                       Love Lesson
34041                         The Dark Side of the Moon
34048                                   Fuck for Forest
34081                                    Frozen Silence
34093                            Deadlier Than the Male
34130  Melhores do Mundo - Hermanoteu na Terra de Godah
34140                          The Education of 

Next, let's look at the ratings dataframe.


In [ ]:
#Drop removes a specified row or column from a dataframe
ratings_df = ratings_df.drop('timestamp', axis=1)
ratings_df.head()